# 1 - Configuração do ambiente

In [ ]:
import os
from pyspark.sql import SparkSession

def get_spark_session(app_name="Hackathon_Project"):

    os.environ["OCI_IAM_TYPE"] = "resource_principal"


    spark = SparkSession.builder \
        .appName(app_name) \
        .config("spark.driver.memory", "20g") \
        .config("spark.executor.memory", "20g") \
        .config("spark.driver.maxResultSize", "4g") \
        .config("spark.hadoop.fs.oci.client.auth.kind", "resource_principal") \
        .config("spark.hadoop.fs.oci.client.regionCodeOrId", "us-chicago-1") \
        .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
        .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY") \
        .config("spark.sql.debug.maxToStringFields", "100") \
        .getOrCreate()


    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    hadoop_conf.set("fs.oci.client.auth.kind", "resource_principal")
    hadoop_conf.set("fs.oci.client.regionCodeOrId", "us-chicago-1")
    hadoop_conf.set("fs.oci.client.custom.authenticator",
                    "com.oracle.bmc.hdfs.auth.ResourcePrincipalsCustomAuthenticator")

    return spark


spark = get_spark_session("Squad_06_Book_Recarga")

# 2 - Bibliotecas

In [ ]:
import os
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, date

from pyspark.sql.functions import col, count, when, isnan, countDistinct, approx_count_distinct, concat, col, lit, substring
from pyspark.sql.types import DoubleType, StringType, NumericType
from pyspark.sql.types import *
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# 3 - Funções

In [ ]:
# --------------------------
# Função para retornar o shape
# --------------------------

def get_shape(dataframe):
    linhas = f'Quanitade de linhas: {dataframe.count()}'
    colunas = f'Quanitade de Colunas: {len(dataframe.columns)}'
    return linhas, colunas

In [ ]:
# --------------------------
# Função para retornar tabela agrupada e percentual
# --------------------------
def Freq(pTabela,pColuna):

    qtd_total=pTabela.count()

    pTabela.registerTempTable("tab_input")
    frq = spark.sql(
            """
                select
                    {col},
                    count(*) as qtd_absoluto,
                    round(100*(count(*) / {tot}),2) as qtd_percentual
                from
                    tab_input
                group by
                    {col}
                order by
                    2 desc
            """.format(col=pColuna, tot=qtd_total))

    qtd=frq.count()
    print('Quantidade de dominios',qtd)
    if qtd > 500:
        frq.show(100,truncate=False)
        return "Dominio muito granular"

    else:
        frq.show(qtd,truncate=False)
        print("volumetria total:",qtd_total)
        return 'Freq da coluna ' + pColuna;

In [ ]:
# --------------------------
# Função para retornar quantidade e porcentagem NULLs e NaNs
# --------------------------
def ver_nulos(dataframe):
  """
  Retorna um DataFrame Pandas ordenado com a contagem e porcentagem d
  e nulos e NaNs Performance: O(1) Action (apenas um scan na tabela).
  """
  pd.set_option('display.max_rows', 100)
  expressoes = []

  for nome_coluna, tipo_coluna in dataframe.dtypes:
      # Verifica se é float/double para checar também NaN (Not a Number)
      if tipo_coluna in ['double', 'float']:
          condicao = (F.col(nome_coluna).isNull() | isnan(F.col(nome_coluna)))
      else:
          condicao = F.col(nome_coluna).isNull()

      expressoes.append(F.count(F.when(condicao, nome_coluna)).alias(nome_coluna))

  # 2. Executa a action do Spark e converte para Pandas
  resultado = dataframe.select(expressoes).toPandas()

  # 3. Transpõe o resultado para formato de tabela (Colunas viram índices)
  resultado_final = resultado.T.rename(columns={0: 'qtd_nulos'})

  # 4. Calcula a porcentagem diretamente no Pandas
  resultado_final['pct_nulos %'] = round((resultado_final['qtd_nulos'] / dataframe.count()) * 100, 2)

  return resultado_final.sort_values('qtd_nulos', ascending=False)

# 4 - Carregando os Dados

In [ ]:
# Definimos o caminho base do seu bucket de prata
base_uri = "oci://layer-silver@axshbddfc2lf/"

# Lista atualizada com os nomes REAIS das pastas no seu Bucket (usando hífen)
# Note que removi as tabelas de pagamento, score e telco conforme solicitado
pastas_alvo = [
    'tabela-bi-dim-canal-aquisicao-credito',
    'tabela-bi-dim-forma-pagamento',
    'tabela-bi-dim-instituicao',
    'tabela-bi-dim-plano-preco',
    'tabela-bi-dim-plataforma',
    'tabela-bi-dim-promocao-credito',
    'tabela-bi-dim-status-plataforma',
    'tabela-bi-dim-tecnologia',
    'tabela-bi-dim-tipo-credito',
    'tabela-bi-dim-tipo-insercao',
    'tabela-bi-dim-tipo-recarga',
    'tabela-cadastral',
    'tabela-recarga'
]

dfs = {}

print('--- 📂 Iniciando Leitura Direta do Object Storage ---')

for tabela in pastas_alvo:
    path_completo = f"{base_uri}{tabela}"
    print(f'📖 Lendo: {tabela}...')

    # O Spark identifica automaticamente as subpastas (partições)
    # de tabela-recarga e tabela-cadastral
    dfs[tabela] = spark.read.parquet(path_completo)

print('\n✅ CARREGAMENTO FINALIZADO')

In [ ]:
recarga = dfs['tabela-recarga']
cadastral = dfs['tabela-cadastral']

# 5 - Inserção das Dimensões

## 5.1 - tabela_bi_bi_dim_status_plataforma

In [ ]:
plataforma = dfs['tabela-bi-dim-status-plataforma'].select('COD_STATUS_PLATAFORMA', 'DSC_STATUS_PLATAFORMA', 'IND_ATIVO')

In [ ]:
book_recarga = recarga.join(
    plataforma,
    on = 'COD_STATUS_PLATAFORMA',
    how = 'left'
).drop('COD_STATUS_PLATAFORMA')

## 5.2 - 'tabela_bi_dim_canal_aquisicao_credito'

In [ ]:
colunas_canal_aquisição = ['COD_CANAL_AQUISICAO', 'COD_SISTEMA_DW', 'DSC_CANAL_AQUISICAO','COD_AGENTE_CREDITO', 'COD_TIPO_CREDITO', 'COD_TIPO_INSTITUICAO']
canal_aquisicao = dfs['tabela-bi-dim-canal-aquisicao-credito'].select(colunas_canal_aquisição)

In [ ]:
canal_aquisicao = canal_aquisicao.withColumns({
    'COD_TIPO_INSTITUICAO': F.when(F.col('COD_CANAL_AQUISICAO') == -3, -3)
                             .when(F.col('COD_CANAL_AQUISICAO') == -2, -2)
                             .otherwise(F.col('COD_TIPO_INSTITUICAO')),

    'COD_AGENTE_CREDITO': F.when(F.col('COD_CANAL_AQUISICAO') == -2, 'NAO DECLARADO')
                           .otherwise(F.col('COD_AGENTE_CREDITO')),

    'COD_TIPO_CREDITO': F.when(F.col('COD_CANAL_AQUISICAO') == -2, 'NAO DECLARADO')
                         .otherwise(F.col('COD_TIPO_CREDITO'))
})

In [ ]:
book_recarga = book_recarga.join(
    canal_aquisicao,
    on = 'COD_CANAL_AQUISICAO',
    how = 'left'
).drop('COD_CANAL_AQUISICAO')

## 5.3 - 'tabela_bi_dim_forma_pagamento'

In [ ]:
forma_pagamento = ['DW_FORMA_PAGAMENTO', 'DSC_FORMA_PAGAMENTO']
df_forma_pagamento = dfs['tabela-bi-dim-forma-pagamento'].select(forma_pagamento)

In [ ]:
book_recarga = book_recarga.join(
    df_forma_pagamento,
    on = 'DW_FORMA_PAGAMENTO',
    how = 'left'
).drop('DW_FORMA_PAGAMENTO')

## 5.4 - 'tabela_bi_dim_instituicao'

In [ ]:
col_instituição = ['DW_INSTITUICAO', 'DSC_INSTITUICAO', 'COD_TIPO_INSTITUICAO', 'DSC_TIPO_INSTITUICAO']
df_instituicao = dfs['tabela-bi-dim-instituicao'].select(col_instituição)

In [ ]:
novos = [
    (-1, 'NAO SE APLICA', -1, 'NAO SE APLICA'),
    (-2, 'NAO DETERMINADO', -2, 'NAO DETERMINADO' )
]
df_novas_linhas = spark.createDataFrame(novos, schema=df_instituicao.schema)
df_instituicao = df_instituicao.union(df_novas_linhas)

In [ ]:
book_recarga = book_recarga.join(
    df_instituicao,
    on = 'DW_INSTITUICAO',
    how = 'left'
).drop('DW_INSTITUICAO')

## 5.5 - 'tabela_bi_dim_plano_preco'

In [ ]:
colunas_plano = ['DW_PLANO', 'DSC_PLANO_PRECO','COD_TIPO_CLIENTE', 'COD_SUB_TIPO_CLIENTE', 'DSC_PLANO_PRECO_BI', 'DSC_TIPO_PLANO_BI',
                 'IND_AMDOCS_PLAT_PRE', 'COD_TRATAMENTO_ESPECIAL', 'COD_SISTEMA_DW', 'DSC_PLANO_PRECO_UNICO_BI', 'COD_PLANO_COMPONENTE']

df_plano = dfs['tabela-bi-dim-plano-preco'].select(colunas_plano)

In [ ]:
book_recarga = book_recarga.join(
    df_plano,
    on =  book_recarga['DW_PLANO_TARIFACAO'] == df_plano["DW_PLANO"],
    how = 'left'
).drop(book_recarga['DW_PLANO_TARIFACAO'])

## 5.6 - 'tabela_bi_dim_plataforma'

In [ ]:
colunas_plataforma = ['DSC_PLATAFORMA','DSC_PLATAFORMA_BI', 'DSC_GRUPO_PLATAFORMA']
df_plataforma = dfs['tabela-bi-dim-plataforma'].select(colunas_plataforma)

In [ ]:
book_recarga = book_recarga.join(
    df_plataforma,
    on = book_recarga['COD_PLATAFORMA_ATU'] == df_plataforma['DSC_PLATAFORMA'],
    how = 'left'
).drop(book_recarga['COD_PLATAFORMA_ATU'])

## 5.7 - 'tabela_bi_dim_tipo_credito'

In [ ]:
colunas_tipo_credito = ['COD_TIPO_CREDITO', 'DSC_TIPO_CREDITO']
df_tipo_credito = dfs['tabela-bi-dim-tipo-credito'].select(colunas_tipo_credito)

In [ ]:
book_recarga = book_recarga.join(
    df_tipo_credito,
    on = 'COD_TIPO_CREDITO',
    how = 'left'
).drop('COD_TIPO_CREDITO')

## 5.8 - 'tabela_bi_dim_tipo_insercao'

In [ ]:
df_insercao = dfs['tabela-bi-dim-tipo-insercao'].select('DW_TIPO_INSERCAO', 'DSC_TIPO_INSERCAO')

In [ ]:
book_recarga = book_recarga.join(
    df_insercao,
    on = 'DW_TIPO_INSERCAO',
    how = 'left'
).drop('DW_TIPO_INSERCAO')

## 5.9 - 'tabela_bi_dim_tipo_recarga'

In [ ]:
df_recarga = dfs['tabela-bi-dim-tipo-recarga'].select('DW_TIPO_RECARGA', 'DSC_TIPO_RECARGA')

In [ ]:
book_recarga = book_recarga.join(
    df_recarga,
    on = 'DW_TIPO_RECARGA',
    how = 'left'
).drop('DW_TIPO_RECARGA')

# 6 - Feature Engineering

In [ ]:
book_recarga = book_recarga.withColumn(

    "HORA_FORMATADA",
    F.lpad(F.col("HOR_INSERCAO_CREDITO").cast("string"), 6, "0")
).withColumn(

    "HORA_INT",
    F.substring(F.col("HORA_FORMATADA"), 1, 2).cast("int")
).withColumn(

    "TURNO_RECARGA",
    F.when((F.col("HORA_INT") >= 0) & (F.col("HORA_INT") < 6), "MADRUGADA")
     .when((F.col("HORA_INT") >= 6) & (F.col("HORA_INT") < 12), "MANHA")
     .when((F.col("HORA_INT") >= 12) & (F.col("HORA_INT") < 18), "TARDE")
     .otherwise("NOITE")
).drop("HORA_FORMATADA", "HORA_INT")


+--------------------+-------------+
|HOR_INSERCAO_CREDITO|TURNO_RECARGA|
+--------------------+-------------+
|              152118|        TARDE|
|              134059|        TARDE|
|               62214|        MANHA|
|               95921|        MANHA|
|              110424|        MANHA|
|              174928|        TARDE|
|              234759|        NOITE|
|               83657|        MANHA|
|              194751|        NOITE|
|              180636|        NOITE|
+--------------------+-------------+
only showing top 10 rows


In [ ]:
# Criar tipo de canal de aquisição de crédito

# 1. Definindo as "palavras-chave" (Regex) para cada grupo
# A tag (?i) no início significa "Case Insensitive" (ignora maiúsculas/minúsculas)
regex_bancos = "(?i)CEF/Banco|CEF/C\\.Bancario|Bradesco|Banco do Brasil|Santander|Itau|Banrisul|Tribanco"
regex_fintechs = "(?i)Mercado Pago|Banco Inter|Recarga Pay|CPay|Claro Pay"
regex_varejo = "(?i)Lotericas|\\[RD\\]|Lojas Claro"
regex_gateways = "(?i)Epay|M4U|Bemobi|DBR|RV\\s|Tendencia|PagSeguro|Log Express|Rede ?flex|Card\\s|RCA\\s|CONEKTA|Globell|POS-|Get Net|Diga|REDE/|Telecom Net|QIWI|MULTILINK|LOG\\s"

# 2. Aplicando a regra de classificação
book_recarga = book_recarga.withColumn(
    "TIPO_CANAL_AQUISICAO",
    F.when(F.col("DSC_CANAL_AQUISICAO") == "NI", "NAO INFORMADO")
     .when(F.col("DSC_CANAL_AQUISICAO") == "ND", "NAO DECLARADO")

     # Se a string contiver o nome de algum banco tradicional:
     .when(F.col("DSC_CANAL_AQUISICAO").rlike(regex_bancos), "BANCOS TRADICIONAL")

     # Se contiver carteiras digitais (inclui o RD CPay da Raia Drogasil):
     .when(F.col("DSC_CANAL_AQUISICAO").rlike(regex_fintechs), "FINTECH OU CARTEIRA DIGITAL")

     # Loterias, lojas físicas e o PDV (Ponto de Venda) físico da Raia Drogasil [RD]:
     .when(F.col("DSC_CANAL_AQUISICAO").rlike(regex_varejo), "VAREJO FÍSICO / DINHEIRO VIVO")

     # Integradores B2B, Maquininhas e Redes de Distribuição (RV, M4U, Redeflex, Log Express):
     .when(F.col("DSC_CANAL_AQUISICAO").rlike(regex_gateways), "GATEWAYS / CAMPANHAS")

     # Qualquer outra coisa que escapar (Portal de Recargas, AGENTE, etc):
     .otherwise("OUTROS")
)

In [ ]:
# Clusterizando plano preço

# 1. Definindo as expressões regulares (ignorando maiúsculas/minúsculas com (?i))
regex_prezao = "(?i)PREZAO"
regex_controle = "(?i)CONTROLE|CONT\\d" # Pega CONTROLE ou CONT seguido de número (ex: Cont35)
regex_jonava = "(?i)JONAVA|ZB\\d"
regex_legado = "(?i)TODA HORA|TH\\s|FALA MAIS|FMB|RECARREGUE"

# 2. Aplicando a clusterização
book_recarga = book_recarga.withColumn(
    "CLUSTER_PLANO",
    F.when(F.col("DSC_PLANO_PRECO") == "-4", "DESCONHECIDO")
     .when(F.col("DSC_PLANO_PRECO").rlike(regex_prezao), "PRE-PAGO CORE")
     .when(F.col("DSC_PLANO_PRECO").rlike(regex_controle), "PLANO CONTROLE")
     .when(F.col("DSC_PLANO_PRECO").rlike(regex_jonava), "PRE-PAGO REGIONAL/CAMPANHA")
     .when(F.col("DSC_PLANO_PRECO").rlike(regex_legado), "PRE-PAGO LEGADO")
     .otherwise("NICHO/OUTROS")
)



+--------------------------+--------+
|CLUSTER_PLANO             |QTD     |
+--------------------------+--------+
|PRE-PAGO CORE             |62799920|
|PLANO CONTROLE            |28462827|
|PRE-PAGO LEGADO           |4879407 |
|DESCONHECIDO              |2152012 |
|PRE-PAGO REGIONAL/CAMPANHA|1908044 |
|NICHO/OUTROS              |11441   |
+--------------------------+--------+



In [ ]:
book_recarga.show()

+-----------+----------+--------------------+--------------------+--------------+-----------------+------------+--------------------+---------+--------+--------------------+----------------+--------------------+--------+---------+--------------+---------------------+---------+--------------+--------------------+------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------+--------------------+----------------+--------------------+------------------+-----------------+-------------------+-----------------------+--------------+------------------------+--------------------+--------------+-----------------+--------------------+----------------+-----------------+----------------+--------------------+--------------------+-------------+
|    NUM_CPF|DW_NUM_NTC|DAT_INSERCAO_CREDITO|HOR_INSERCAO_CREDITO|DW_NUM_CLIENTE|COD_TECNOLOGIA_DW|COD_PROMOCAO|VAL_CREDITO_INSERIDO|VAL_BONUS|VAL_REAl|IND_METODO_PAGAMENTO|COD_GRUPO_CARTAO|D

# 7 - DF para join

In [ ]:
recarga_select = ['NUM_CPF', 'DW_NUM_NTC', 'DAT_INSERCAO_CREDITO', 'VAL_CREDITO_INSERIDO', 'FLAG_SOS', 'VALOR_SOS', 'COD_GRUPO_CARTAO',
                  'DSC_GRUPO_CARTAO_WPP', 'DSC_FORMA_PAGAMENTO', 'DSC_TIPO_INSTITUICAO', 'DSC_PLATAFORMA_BI', 'DSC_GRUPO_PLATAFORMA',
                  'DSC_TIPO_RECARGA', 'TURNO_RECARGA', 'TIPO_CANAL_AQUISICAO', 'CLUSTER_PLANO']

In [ ]:
book_recarga_sel = book_recarga.select(recarga_select)

# 8 - Join

In [ ]:
# ==============================================================================
# CAMADA 1: PREPARAÇÃO DA ESPINHA (SPINE) E CRUZAMENTO ZERO-LEAKAGE
# ==============================================================================

# 1. Preparar a data de corte da Safra
df_spine = cadastral.select("ID_UNICO", "NUM_CPF", "SAFRA").withColumn(
    "DATA_CORTE_SAFRA",
    F.to_date(F.col("SAFRA").cast("string"), "yyyyMM")
)

# 2. Join e Filtro Anti-Leakage IMPLACÁVEL
# Mantemos apenas recargas estritamente anteriores à Safra daquele ID_UNICO
df_base = df_spine.join(book_recarga_sel, on="NUM_CPF", how="inner") \
    .filter(F.col("DAT_INSERCAO_CREDITO") < F.col("DATA_CORTE_SAFRA"))

In [ ]:
# ==============================================================================
# CAMADA 2: DATA QUALITY & REGRAS DE NEGÓCIO (LINHA A LINHA)
# ==============================================================================

# 1. Regra de Negócio: Taxa SOS e Valor Real
df_base = df_base.withColumn(
    "TAXA_SOS",
    F.when(F.col("VALOR_SOS") == 0, 0)
     .when(F.col("VALOR_SOS") <= 3, 1.0)
     .when(F.col("VALOR_SOS") <= 5, 2.0)
     .when(F.col("VALOR_SOS") <= 10, 3.5)
     .when(F.col("VALOR_SOS") <= 15, 5.0)
     .when(F.col("VALOR_SOS") <= 20, 6.0)
     .otherwise(7.0) # Margem de segurança caso haja valor maior
).withColumn(
    "VAL_REAL_RECARGA",
    F.col("VAL_CREDITO_INSERIDO") - F.col("VALOR_SOS") - F.col("TAXA_SOS")
)

# 2. Distância Temporal (Dias entre a recarga e a Safra)
df_base = df_base.withColumn(
    "DIAS_ATE_SAFRA",
    F.datediff(F.col("DATA_CORTE_SAFRA"), F.col("DAT_INSERCAO_CREDITO"))
)

# 3. Tratamento de Categorias (Redução de Cardinalidade)
# Exemplo para Grupo Cartão (Top 1%)
top_cartoes = ['UB','WT','PY','I8','UD','IW','UC','FW','FV','G6','AX','I4','OR','IC','IB','PZ','IA','N9','UE']
top_wpp = ['NaoSeAplica', 'Rec.Online', 'AtivPromocao', 'ChipPre+R$30', 'ForcaZB2']

df_base = df_base.withColumn(
    "COD_GRUPO_CARTAO_TRAT",
    F.when(F.col("COD_GRUPO_CARTAO") == "-2", "NAO_DETERMINADO")
     .when(F.col("COD_GRUPO_CARTAO").isin(top_cartoes), F.col("COD_GRUPO_CARTAO"))
     .otherwise("OUTROS")
).withColumn(
    "DSC_GRUPO_CARTAO_WPP_TRAT",
    F.when(F.col("DSC_GRUPO_CARTAO_WPP") == "-2", "NAO_DETERMINADO")
     .when(F.col("DSC_GRUPO_CARTAO_WPP").isin(top_wpp), F.col("DSC_GRUPO_CARTAO_WPP"))
     .otherwise("OUTROS")
).withColumn(
    "DSC_TIPO_INSTITUICAO_TRAT",
    F.when(F.col("DSC_TIPO_INSTITUICAO").isNull(), "OUTROS").otherwise(F.col("DSC_TIPO_INSTITUICAO"))
)


In [ ]:
# ==============================================================================
# CAMADA 3: FEATURE ENGINEERING (AGREGAÇÃO POR ID_UNICO) - COM ARREDONDAMENTOS
# ==============================================================================

book_recarga_final = df_base.groupBy("ID_UNICO", "NUM_CPF", "SAFRA").agg(

    # --- R: RECÊNCIA E FREQUÊNCIA ---
    F.min("DIAS_ATE_SAFRA").alias("QTD_DIAS_ULTIMA_RECARGA"),
    F.count("*").alias("QTD_RECARGAS_HIST"),
    F.countDistinct("DW_NUM_NTC").alias("QTD_CELULARES_RECARREGADOS"),

    F.sum(F.when(F.col("DIAS_ATE_SAFRA") <= 30, 1).otherwise(0)).alias("QTD_RECARGAS_U30D"),
    F.sum(F.when((F.col("DIAS_ATE_SAFRA") > 30) & (F.col("DIAS_ATE_SAFRA") <= 90), 1).otherwise(0)).alias("QTD_RECARGAS_U31_90D"),

    # --- M: MONETÁRIO (Com as novas colunas e arredondamentos) ---
    F.round(F.sum("VAL_CREDITO_INSERIDO"), 2).alias("VAL_TOTAL_CREDITO_HIST"),
    F.round(F.max("VAL_CREDITO_INSERIDO"), 2).alias("VAL_MAX_RECARGA"),
    F.round(F.avg("VAL_CREDITO_INSERIDO"), 2).alias("VAL_MEDIA_RECARGA"),

    F.round(F.sum("VAL_REAL_RECARGA"), 2).alias("VAL_TOTAL_REAL_HIST"),
    F.round(F.avg("VAL_REAL_RECARGA"), 2).alias("VAL_TICKET_MEDIO_REAL"),

    # --- S: SOS E RISCO ---
    F.sum("FLAG_SOS").alias("QTD_USO_SOS_HIST"),
    F.round(F.sum("VALOR_SOS"), 2).alias("VAL_TOTAL_SOS_TOMADO"),
    F.round(F.sum("TAXA_SOS"), 2).alias("VAL_TOTAL_TAXA_SOS_PAGA"),

    # NOVO: Média do SOS apenas quando o cliente USOU o SOS (Ticket Médio do SOS)
    # Se dividirmos por todas as recargas, o valor fica distorcido. Dividir pelo uso é mais preciso.
    F.round(
        F.when(F.sum("FLAG_SOS") > 0, F.sum("VALOR_SOS") / F.sum("FLAG_SOS")).otherwise(0), 2
    ).alias("VAL_MEDIA_SOS_TOMADO"),

    # --- JANELAS TEMPORAIS ---
    F.round(F.sum(F.when(F.col("DIAS_ATE_SAFRA") <= 30, F.col("VAL_REAL_RECARGA")).otherwise(0)), 2).alias("VAL_REAL_U30D"),
    F.round(F.sum(F.when((F.col("DIAS_ATE_SAFRA") > 30) & (F.col("DIAS_ATE_SAFRA") <= 90), F.col("VAL_REAL_RECARGA")).otherwise(0)), 2).alias("VAL_REAL_U31_90D")
)


In [ ]:
# ==============================================================================
# CAMADA 4: CRIAÇÃO DE FEATURES DERIVADAS (RATIOS)
# ==============================================================================

book_recarga_final = book_recarga_final.withColumn(
    "VAL_MEDIA_DIAS_ENTRE_RECARGAS",
    F.round(
        F.when(F.col("QTD_RECARGAS_HIST") > 1,
               F.col("QTD_DIAS_ULTIMA_RECARGA") / F.col("QTD_RECARGAS_HIST")).otherwise(F.col("QTD_DIAS_ULTIMA_RECARGA")),
        2
    )
).withColumn(
    "PCT_MOMENTUM_RECARGA_30D",
    F.round(
        F.when(F.col("VAL_REAL_U31_90D") > 0, F.col("VAL_REAL_U30D") / F.col("VAL_REAL_U31_90D")).otherwise(0),
        2
    )
).withColumn(
    "PCT_DEPENDENCIA_SOS",
    F.round(
        F.when(F.col("VAL_TOTAL_CREDITO_HIST") > 0, F.col("VAL_TOTAL_SOS_TOMADO") / F.col("VAL_TOTAL_CREDITO_HIST")).otherwise(0),
        4 # Mantive 4 casas aqui pois é um percentual (ex: 0.0345 = 3.45%)
    )
)

In [ ]:
# ==============================================================================
# CAMADA 5: PIVOTEAMENTO DAS CATEGORIAS (BLINDADO CONTRA CARACTERES ESPECIAIS)
# ==============================================================================

def criar_pivot(df_origem, coluna_pivot, prefixo):
    """Agrupa por ID_UNICO, pivota e renomeia limpando caracteres especiais"""

    # 1. Agrupar e Pivotar
    df_pivoted = df_origem.groupBy("ID_UNICO").pivot(coluna_pivot).agg(F.count(F.col(coluna_pivot)))

    cols_to_fill = []

    # 2. Renomear e Limpar Nomes
    for col_name in df_pivoted.columns:
        if col_name != "ID_UNICO":

            # Mágica do Regex: Substitui TUDO que não for letra ou número por '_'
            # Ex: 'Rec.Online' vira 'Rec_Online'. 'ChipPre+R$30' vira 'ChipPre_R_30'
            clean_name = re.sub(r'[^a-zA-Z0-9]', '_', col_name).upper()

            # Remove underlines duplicados (ex: '__' vira '_') e tira do final
            clean_name = re.sub(r'_+', '_', clean_name).strip('_')

            new_col_name = f"QTD_{prefixo}_{clean_name}"

            # Aplica a renomeação
            df_pivoted = df_pivoted.withColumnRenamed(col_name, new_col_name)
            cols_to_fill.append(new_col_name)

    # 3. Preencher Nulos com 0
    df_pivoted = df_pivoted.na.fill(0, subset=cols_to_fill)

    return df_pivoted

# ==============================================================================

# Agora pode rodar as linhas abaixo normalmente!
df_turno = criar_pivot(df_base, "TURNO_RECARGA", "TURNO")
df_canal = criar_pivot(df_base, "TIPO_CANAL_AQUISICAO", "CANAL")
df_plano = criar_pivot(df_base, "CLUSTER_PLANO", "PLANO")
df_cartao = criar_pivot(df_base, "COD_GRUPO_CARTAO_TRAT", "CARTAO")
df_wpp = criar_pivot(df_base, "DSC_GRUPO_CARTAO_WPP_TRAT", "ORIGEM")
df_plat = criar_pivot(df_base, "DSC_PLATAFORMA_BI", "PLATAFORMA")
df_tipo_recarga = criar_pivot(df_base, "DSC_TIPO_RECARGA", "TIPO_RECAR")

# O Grande Join (Exatamente como estava no passo anterior)
book_recarga_final_com_dummies = book_recarga_final \
    .join(df_turno, on="ID_UNICO", how="left") \
    .join(df_canal, on="ID_UNICO", how="left") \
    .join(df_plano, on="ID_UNICO", how="left") \
    .join(df_cartao, on="ID_UNICO", how="left") \
    .join(df_wpp, on="ID_UNICO", how="left") \
    .join(df_plat, on="ID_UNICO", how="left") \
    .join(df_tipo_recarga, on="ID_UNICO", how="left")

colunas_dummies = [c for c in book_recarga_final_com_dummies.columns if c.startswith("QTD_")]
book_recarga_final_com_dummies = book_recarga_final_com_dummies.na.fill(0, subset=colunas_dummies)


In [ ]:
path_gold_recarga = "oci://layer-gold@axshbddfc2lf/book-recarga"
book_recarga_final_com_dummies.write.mode("overwrite").partitionBy("SAFRA").parquet(path_gold_recarga)